# Cognee + Claude Agent SDK Integration Guide

This notebook demonstrates how to integrate **Cognee** (a semantic memory for AI agents) with **Claude Agent SDK** to create AI agents with persistent, cross-session memory capabilities powered by knowledge graphs + embeddings.

## What is Cognee?

**Cognee** is an open-source semantic memory layer that transforms unstructured, structured, semi-structured data into queryable knowledge graphs backed by embeddings. Cognee:

- **Automatically extracts** entities, relationships, and semantic meaning from text
- **Creates knowledge graphs** with embeddings 
- **Enables natural language queries** to retrieve relevant information
- **Maintains context** across different sessions and interactions
- **Supports multi-modal data** including text, documents, and structured data

## What is Claude Agent SDK? 

**Claude Agent SDK** is Anthropic's framework for building AI agents with Claude. It provides:

- **ClaudeSDKClient**: Easy-to-use client for interacting with Claude
- **Tool Integration**: Seamless integration with custom tools via MCP servers
- **Message Types**: Structured message types (UserMessage, AssistantMessage, etc.)
- **Streaming Support**: Real-time response streaming

## Why Combine Them?

The Cognee-Claude integration provides AI agents with persistent, semantic memory that survives between sessions. Agents can store information in Cognee's knowledge graph and retrieve it using natural language queries, enabling more accurate retrieval, continuity across conversations without manual state management.

Let's explore how this works in practice!


## 📋 Prerequisites

Before running this notebook, make sure you have the required dependencies installed:

```bash
# Using uv
uv sync

# Or using pip
pip install cognee-integration-claude
```


## Environment Setup

Both Claude Agent SDK and Cognee require API keys for LLM operations. Let's configure the environment:


In [ ]:
import os

from dotenv import load_dotenv

# Load .env before reading os.getenv below.
load_dotenv()

# cognee's LLM key; other providers: https://docs.cognee.ai/setup-configuration/llm-providers
if not os.getenv("LLM_API_KEY"):
    raise RuntimeError(
        "LLM_API_KEY is not set. Add it to a .env file in this directory "
        "(LLM_API_KEY=sk-...) or export it before launching the kernel."
    )

# The bundled Claude Code CLI handles auth — no ANTHROPIC_API_KEY needed.

## Claude Agent SDK Fundamentals

Before diving into the Cognee integration, let's understand Claude Agent SDK's core building blocks:

### ClaudeSDKClient
**ClaudeSDKClient** is the central abstraction in Claude Agent SDK. It encapsulates:
- Connection to Claude via Anthropic's API
- MCP server configuration for tools
- Tool permissions (allowed/disallowed)

### ClaudeAgentOptions
**ClaudeAgentOptions** configures the client:
- **mcp_servers**: Dictionary of MCP servers providing tools
- **allowed_tools**: Tools the agent can use
- **disallowed_tools**: Tools to disable

### Tools
**Tools** are functions that agents can call. Claude Agent SDK uses:
- **@tool decorator**: For creating async tool functions
- **create_sdk_mcp_server**: For bundling tools into MCP servers

### Message Types
The SDK provides structured message types:
- **UserMessage**: Messages from the user
- **AssistantMessage**: Responses from Claude
- **SystemMessage**: System-level messages
- **ResultMessage**: Final result messages with metadata

Let's create a simple setup to understand the basics:


In [ ]:
from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ClaudeSDKClient,
    ResultMessage,
    SystemMessage,
    TextBlock,
    UserMessage,
    create_sdk_mcp_server,
)


def display_message(msg):
    """Standardized message display function.

    - UserMessage: "User: <content>"
    - AssistantMessage: "Claude: <content>"
    - SystemMessage: ignored
    - ResultMessage: "Result ended" + cost if available
    """
    if isinstance(msg, UserMessage):
        for block in msg.content:
            if isinstance(block, TextBlock):
                print(f"User: {block.text}")
    elif isinstance(msg, AssistantMessage):
        for block in msg.content:
            if isinstance(block, TextBlock):
                print(f"Claude: {block.text}")
    elif isinstance(msg, SystemMessage):
        pass
    elif isinstance(msg, ResultMessage):
        print("Result ended")


print("✓ Helper functions defined")

## Introducing Cognee to Claude: Semantic Memory for Agents

Now let's add **persistent semantic memory** to our agents using Cognee!

### What Cognee Adds to Claude:

- **Semantic Memory**: Store and retrieve information using natural language from Cognee's knowledge graph backed by embeddings
- **Knowledge Graphs**: Automatic entity and relationship extraction
- **Cross-Session Persistence**: Memory survives between different agent instances
- **Intelligent Search**: Find relevant information by meaning using Cognee's advanced retrieval methods
- **Two memory tiers**: a fast session cache and a permanent knowledge graph (more below)

### Cognee Tools Integration

`cognee_tools()` builds two MCP tools (cognee's v1.0 memory API):
- **`remember`**: Store information in Cognee's knowledge graph (`cognee.remember`)
- **`recall`**: Query stored information using natural language (`cognee.recall`)

Pass `cognee_tools(session_id=...)` to route writes through cognee's session cache.

Let's start fresh by clearing any existing Cognee data:


In [ ]:
import cognee
from cognee.api.v1.config import config

# Isolated dirs so forget(everything=True) can't wipe real cognee data.
config.data_root_directory(os.path.join(os.getcwd(), "../.cognee/data_storage"))
config.system_root_directory(os.path.join(os.getcwd(), "../.cognee/system"))

await cognee.forget(everything=True)

print("✓ Cognee memory cleared successfully!")

### Importing Cognee Tools

Now let's import the Cognee tools and create an agent with semantic memory capabilities:


In [ ]:
from cognee_integration_claude import cognee_tools

cognee_server = create_sdk_mcp_server(name="cognee-tools", version="1.0.0", tools=cognee_tools())

memory_options = ClaudeAgentOptions(
    mcp_servers={"tools": cognee_server},
    allowed_tools=["mcp__tools__remember", "mcp__tools__recall"],
    disallowed_tools=[
        "Task",
        "Bash",
        "Glob",
        "Grep",
        "ExitPlanMode",
        "Read",
        "Edit",
        "Write",
        "NotebookEdit",
        "WebFetch",
        "TodoWrite",
        "WebSearch",
        "BashOutput",
        "KillShell",
        "SlashCommand",
    ],
)

print("✓ Memory agent configured with Cognee tools: remember + recall")

### Storing Information with Cognee

Let's give our agent some contract information to remember. The agent will use the `remember` tool to store this in Cognee's knowledge graph:


In [ ]:
contracts_data = """
We have signed a contract with the following company: "Meditech Solutions". Company is in the healthcare industry. Start date is Jan 2023 and end date is Dec 2025. Contract value is £1.2M.
We have signed a contract with the following company: "QuantumSoft". Company is in the technology industry. Start date is Aug 2024 and end date is Aug 2028. Contract value is £5.5M.
We have signed a contract with the following company: "Orion Retail Group". Company is in the retail industry. Start date is Mar 2024 and end date is Mar 2026. Contract value is £850K.
"""

print("=== ADDING CONTRACTS TO MEMORY ===")
async with ClaudeSDKClient(options=memory_options) as client:
    await client.query(f"Please remember this contract information: {contracts_data}")

    async for msg in client.receive_response():
        display_message(msg)

print("\n✓ Contract data sent to Cognee memory!")

### Background Data Ingestion

Let's also add some additional data directly to Cognee (bypassing the agent) to demonstrate how the agent can access pre-existing knowledge. This simulates a scenario where:
- **Historical data** exists in the knowledge base
- **Agent interactions** add new information
- **Both sources** are searchable together


In [ ]:
additional_contracts = """
We have signed a contract with the following company: "HealthBridge Systems". Company is in the healthcare industry. Start date is Feb 2023 and end date is Jan 2026. Contract value is £2.4M.
We have signed a contract with the following company: "NovaCare Diagnostics". Company is in the healthcare industry. Start date is May 2024 and end date is Apr 2027. Contract value is £1.6M.
We have signed a contract with the following company: "SkyPort Logistics". Company is in the logistics industry. Start date is Nov 2022 and end date is Oct 2026. Contract value is £3.1M.
"""

print("Adding background data directly to Cognee...")
await cognee.remember(additional_contracts)
print("✓ Background data processed and added to knowledge graph!")

### Visualizing the Knowledge Graph

Let's visualize what Cognee has built from our data:


In [ ]:
import webbrowser

from cognee.api.v1.visualize import visualize_multi_user_graph
from cognee.modules.users.methods import get_default_user


# Plain visualize_graph() can't see per-dataset graphs under cognee's access
# control, so render every dataset the default user owns.
async def visualize_graph(file_name, open_browser=True):
    destination_file_path = os.path.join(os.getcwd(), "../.cognee", file_name)
    user = await get_default_user()
    pairs = [(user, ds) for ds in await cognee.datasets.list_datasets(user=user)]
    await visualize_multi_user_graph(pairs, destination_file_path=destination_file_path)

    if open_browser:
        url = "file://" + os.path.abspath(destination_file_path)
        webbrowser.open(url)

    print(f"✓ Graph visualization saved to: {destination_file_path}")


await visualize_graph(file_name="first_graph_visualization.html")

Open the generated HTML file in your browser to explore the knowledge graph. You'll see:
- Entities extracted from the contracts (companies, industries, values)
- Relationships between entities
- Data from both agent interactions and background ingestion


## Cross-Session Memory Persistence

Now let's demonstrate one of Cognee's key features: **persistent memory across agent instances**.

We'll create a completely fresh agent instance that:
- Has **no conversation history** from the previous agent
- Has **no internal state** carried over
- **CAN access** all information stored in Cognee's knowledge graph

This shows how Cognee provides **true persistent memory** that survives agent restarts:


In [ ]:
search_query = "I need to research our contract portfolio. Can you search for any contracts we have with companies in the healthcare industry?"

print("=== SEARCHING WITH FRESH AGENT ===")
print(f"Query: {search_query}\n")

async with ClaudeSDKClient(options=memory_options) as fresh_client:
    await fresh_client.query(search_query)

    print("=== AGENT RESPONSE ===")
    async for msg in fresh_client.receive_response():
        display_message(msg)

**Amazing!** The fresh agent found all the healthcare contracts, including:
- Contracts added through agent interactions (Meditech Solutions)
- Contracts added through background ingestion (HealthBridge Systems, NovaCare Diagnostics)

This demonstrates true **cross-session persistence** - the knowledge survives agent restarts!


## Session Memory: cache now, persist later

Every `remember` so far wrote straight into Cognee's **permanent knowledge graph**. Cognee also has a faster **session cache** tier, and `cognee_tools` exposes it through one parameter:

- **No `session_id`** → `remember` runs full extraction and writes to the **permanent graph** (what we did above).
- **With `session_id`** → `remember` writes to that session's **cache** (cheap, no graph extraction). It is *not* in the graph yet.
- **`cognee.improve(session_ids=[...])`** → promotes a session's cached entries into the permanent graph.

So an agent can capture things cheaply during a session, then persist the useful parts later.

> We pass `remember_kwargs={"self_improvement": False}` to the session tools so cached writes **stay** in the cache until we explicitly call `improve()`. (By default cognee bridges them to the graph in the background.)

In [ ]:
await cognee.forget(everything=True)


async def ask(options, prompt):
    """Send one prompt to a fresh agent and return its text reply."""
    reply = ""
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt)
        async for msg in client.receive_response():
            if isinstance(msg, AssistantMessage):
                for block in msg.content:
                    if isinstance(block, TextBlock):
                        reply += block.text
    return reply.strip()


permanent_options = ClaudeAgentOptions(
    mcp_servers={
        "tools": create_sdk_mcp_server(name="permanent", version="1.0.0", tools=cognee_tools())
    },
    allowed_tools=["mcp__tools__remember", "mcp__tools__recall"],
)

SESSION_ID = "mission-briefing"
session_options = ClaudeAgentOptions(
    mcp_servers={
        "tools": create_sdk_mcp_server(
            name="session",
            version="1.0.0",
            tools=cognee_tools(session_id=SESSION_ID, remember_kwargs={"self_improvement": False}),
        )
    },
    allowed_tools=["mcp__tools__remember", "mcp__tools__recall"],
)

print("✓ Permanent and session agents ready")

### 1. Seed the permanent memory

The permanent agent stores a baseline fact — it goes straight into the graph.

In [ ]:
print(
    await ask(
        permanent_options,
        'Use your remember tool to store: "Orion Retail Group" — retail industry, contract worth £850K.',
    )
)

### 2. The session agent records an update — into the cache only

This write lands in the session cache, **not** the graph yet.

In [ ]:
print(
    await ask(
        session_options,
        "Use your remember tool to store: The Orion Retail Group contract was renewed "
        "for 3 more years at an increased value of £2.0M for 2026.",
    )
)

### 3. Before `improve()`: the permanent agent can't see the cached update

The permanent agent only searches the graph, where the renewal hasn't landed — so it still reports the old £850K.

In [ ]:
question = (
    "Use your recall tool: what is the renewed 2026 value of the Orion Retail Group contract?"
)
print(await ask(permanent_options, question))

await visualize_graph(file_name="session_before_improve.html")

### 4. Persist the cache with `cognee.improve()`

This promotes the session's cached entries into the permanent knowledge graph.

In [ ]:
await cognee.improve(session_ids=[SESSION_ID])
print(f"✓ Persisted session '{SESSION_ID}' into the permanent graph")

### 5. After `improve()`: now the permanent agent sees it

Same question, same agent — but the renewal is now in the graph, so the answer updates to £2.0M.

In [ ]:
print(await ask(permanent_options, question))

await visualize_graph(file_name="session_after_improve.html")

**What just happened:**

1. The session agent's write landed in the **cache**, invisible to graph search.
2. The permanent agent kept answering with the old value (£850K).
3. `cognee.improve(session_ids=[...])` bridged the cache into the **graph**.
4. The same question now returns the updated value (£2.0M), and `session_after_improve.html` has more nodes than `session_before_improve.html`.

See `examples/session_memory.py` for this same flow as a standalone script.

## Summary

In this guide, we've learned how to:

### Claude Agent SDK Fundamentals
- ✅ Configure agents with `ClaudeAgentOptions`
- ✅ Create tools using the `@tool` decorator
- ✅ Bundle tools into MCP servers with `create_sdk_mcp_server`
- ✅ Execute agents using `ClaudeSDKClient`

### Cognee Integration
- ✅ Use the `remember` tool to store information in the knowledge graph
- ✅ Use the `recall` tool to query stored information
- ✅ Demonstrate cross-session memory persistence
- ✅ Visualize knowledge graphs

### Session Memory
- ✅ Route writes to the session cache with `cognee_tools(session_id=...)`
- ✅ Keep cached writes out of the graph with `remember_kwargs={"self_improvement": False}`
- ✅ Promote a session into the permanent graph with `cognee.improve(session_ids=[...])`

### Key Takeaways

1. **Cognee provides persistent memory** that survives agent restarts
2. **Session memory** captures data cheaply in a cache, then `improve()` persists it to the graph
3. **Knowledge graphs** enable semantic search across stored information
4. **Easy integration** with Claude Agent SDK through `cognee_tools()`

## Next Steps

- Explore advanced Cognee features like custom pipelines
- Integrate with other data sources (documents, PDFs)
- Deploy to production
- Check out the [Cognee documentation](https://docs.cognee.ai) for more capabilities
